In [ ]:
!pip install lightning pytorch-forecasting

In [ ]:
import copy
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

import lightning.pytorch as pl

from lightning.pytorch.callbacks import (
    Callback,
    ModelCheckpoint
)

from pytorch_forecasting import (
    TimeSeriesDataSet,
    TemporalFusionTransformer
)

from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss


# 1. Check that the shared metric function is available
if "calculate_metrics" not in globals():
    raise RuntimeError(
        "Run Code 7 first to define calculate_metrics()."
    )


# 2. Define paths and training settings
input_path = Path(
    "results/step_04/engineered_panel.csv"
)

output_folder = Path(
    "results/step_09"
)

output_folder.mkdir(
    parents=True,
    exist_ok=True
)

SEED = 42
ENCODER_LENGTH = 12
FORECAST_HORIZON = 6

HIDDEN_SIZE = 16
ATTENTION_HEADS = 1
LEARNING_RATE = 0.03
EPOCHS = 100
BATCH_SIZE = 128

TRAIN_END_DATE = pd.Timestamp("2023-12-01")
VALIDATION_END_DATE = pd.Timestamp("2024-06-01")
TEST_END_DATE = pd.Timestamp("2024-12-01")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.set_num_threads(2)

torch.use_deterministic_algorithms(
    True,
    warn_only=True
)


# 3. Load and prepare the engineered panel
panel = pd.read_csv(
    input_path,
    parse_dates=["Entry_Date"]
)

panel["Reference"] = (
    panel["Reference"].astype(str)
)

panel["Quantity"] = (
    panel["Quantity"].astype(float)
)

panel["time_idx"] = (
    panel["time_idx"].astype(int)
)

panel = panel.sort_values(
    ["Reference", "time_idx"]
).reset_index(drop=True)

references = sorted(
    panel["Reference"].unique()
)

n_parts = len(references)

# These variables are available only as historical observations.
unknown_reals = [
    "Quantity",
    "Unit_Margin",
    "Total_Purchase_Cost",
    "Fuzzy_TOPSIS_Score"
]

# Calendar variables are known for future months.
known_reals = [
    "time_idx",
    "month_sin",
    "month_cos"
]


# 4. Create the training dataset
training_panel = panel.loc[
    panel["Entry_Date"] <= TRAIN_END_DATE
].copy()

training_dataset = TimeSeriesDataSet(
    training_panel,
    time_idx="time_idx",
    target="Quantity",
    group_ids=["Reference"],

    min_encoder_length=ENCODER_LENGTH,
    max_encoder_length=ENCODER_LENGTH,

    min_prediction_length=FORECAST_HORIZON,
    max_prediction_length=FORECAST_HORIZON,

    time_varying_known_reals=known_reals,
    time_varying_unknown_reals=unknown_reals,

    target_normalizer=GroupNormalizer(
        groups=["Reference"]
    ),

    add_relative_time_idx=False,
    add_target_scales=False,
    add_encoder_length=False
)


# 5. Create validation and test datasets
# Reuse the encoders and normalizers fitted on training data.
# predict=True selects one final six-month target window per part.
validation_panel = panel.loc[
    panel["Entry_Date"] <= VALIDATION_END_DATE
].copy()

test_panel = panel.loc[
    panel["Entry_Date"] <= TEST_END_DATE
].copy()

validation_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    validation_panel,
    predict=True,
    stop_randomization=True
)

test_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    test_panel,
    predict=True,
    stop_randomization=True
)


# 6. Create the data loaders
training_loader = training_dataset.to_dataloader(
    train=True,
    batch_size=BATCH_SIZE,
    num_workers=0
)

validation_loader = validation_dataset.to_dataloader(
    train=False,
    batch_size=BATCH_SIZE,
    num_workers=0
)

test_loader = test_dataset.to_dataloader(
    train=False,
    batch_size=BATCH_SIZE,
    num_workers=0
)


# 7. Record the losses and validation metrics during training
class TrainingHistory(Callback):

    def __init__(self):
        super().__init__()
        self.records = []

    def on_validation_epoch_end(self, trainer, pl_module):

        if trainer.sanity_checking:
            return

        record = {
            "Epoch": trainer.current_epoch + 1
        }

        for name, value in trainer.callback_metrics.items():

            if (
                isinstance(value, torch.Tensor)
                and value.numel() == 1
            ):
                record[name] = float(
                    value.detach().cpu().item()
                )

        self.records.append(record)

        pd.DataFrame(
            self.records
        ).to_csv(
            output_folder / "TFT_training_history.csv",
            index=False
        )

        if (
            trainer.current_epoch == 0
            or (trainer.current_epoch + 1) % 10 == 0
        ):
            validation_loss = record.get(
                "val_loss",
                float("nan")
            )

            print(
                f"Epoch {trainer.current_epoch + 1:03d}/{EPOCHS} | "
                f"Validation quantile loss: "
                f"{validation_loss:.5f}"
            )


history_callback = TrainingHistory()


# 8. Configure validation-based checkpoint selection
checkpoint_callback = ModelCheckpoint(
    dirpath=str(
        output_folder / "checkpoints"
    ),
    filename="TFT_best",
    monitor="val_loss",
    mode="min",
    save_top_k=1
)


# 9. Define the Temporal Fusion Transformer
tft_model = TemporalFusionTransformer.from_dataset(
    training_dataset,
    learning_rate=LEARNING_RATE,
    hidden_size=HIDDEN_SIZE,
    attention_head_size=ATTENTION_HEADS,
    hidden_continuous_size=8,
    dropout=0.1,

    # Predict three quantiles; use the median for point evaluation.
    output_size=3,
    loss=QuantileLoss(
        quantiles=[0.1, 0.5, 0.9]
    ),

    optimizer="adam",
    log_interval=-1,
    reduce_on_plateau_patience=1000
)


# 10. Configure and run training
trainer = pl.Trainer(
    max_epochs=EPOCHS,
    accelerator="cpu",
    devices=1,
    logger=False,
    enable_model_summary=False,
    enable_progress_bar=False,
    num_sanity_val_steps=0,
    gradient_clip_val=0.1,
    deterministic=True,
    callbacks=[
        history_callback,
        checkpoint_callback
    ]
)

trainer.fit(
    tft_model,
    train_dataloaders=training_loader,
    val_dataloaders=validation_loader
)


# 11. Restore the best validation checkpoint
best_checkpoint_path = (
    checkpoint_callback.best_model_path
)

best_tft = (
    TemporalFusionTransformer.load_from_checkpoint(
        best_checkpoint_path,
        map_location="cpu",
        weights_only=False
    )
)

best_tft.eval()

print(
    "\nBest checkpoint:",
    best_checkpoint_path
)

print(
    "Best validation loss:",
    float(
        checkpoint_callback.best_model_score.item()
    )
)


# 12. Check that unavailable future covariates are excluded
# Future actual quantities, margins, costs, and scores
# must not be used as decoder input variables.
assert not set(
    unknown_reals
).intersection(
    best_tft.decoder_variables
)


# 13. Retrieve the part identifier decoder
group_encoder = (
    training_dataset
    .get_parameters()["categorical_encoders"]
    ["__group_id__Reference"]
)

date_by_time_index = (
    panel[
        ["time_idx", "Entry_Date"]
    ]
    .drop_duplicates()
    .set_index("time_idx")["Entry_Date"]
    .to_dict()
)


# 14. Generate six-month test predictions
prediction_records = []

with torch.no_grad():

    for inputs, targets in test_loader:

        # TFT returns predictions in original target units.
        model_output = best_tft(
            inputs
        )

        quantile_predictions = (
            model_output["prediction"]
        )

        median_predictions = (
            best_tft.loss.to_prediction(
                quantile_predictions
            )
            .cpu()
            .numpy()
        )

        median_predictions = np.maximum(
            median_predictions,
            0
        )

        actual_values = (
            targets[0]
            .cpu()
            .numpy()
        )

        encoded_groups = (
            inputs["groups"][:, 0]
            .cpu()
            .numpy()
        )

        decoded_references = (
            group_encoder.inverse_transform(
                encoded_groups
            )
        )

        decoder_time_indices = (
            inputs["decoder_time_idx"]
            .cpu()
            .numpy()
        )

        for row_index, reference in enumerate(
            decoded_references
        ):

            first_target_index = int(
                decoder_time_indices[
                    row_index,
                    0
                ]
            )

            origin_date = pd.Timestamp(
                date_by_time_index[
                    first_target_index - 1
                ]
            )

            for horizon in range(1, FORECAST_HORIZON + 1):

                target_index = int(
                    decoder_time_indices[
                        row_index,
                        horizon - 1
                    ]
                )

                target_date = pd.Timestamp(
                    date_by_time_index[
                        target_index
                    ]
                )

                prediction_records.append({
                    "Model": "TFT",
                    "Reference": str(reference),
                    "Origin_Date": origin_date.strftime(
                        "%Y-%m-%d"
                    ),
                    "Target_Date": target_date.strftime(
                        "%Y-%m-%d"
                    ),
                    "Horizon": horizon,
                    "Actual": float(
                        actual_values[
                            row_index,
                            horizon - 1
                        ]
                    ),
                    "Prediction": float(
                        median_predictions[
                            row_index,
                            horizon - 1
                        ]
                    ),
                    "Is_Synthetic": True
                })

predictions_df = pd.DataFrame(
    prediction_records
)


# 15. Verify the common test target identities
expected_targets = pd.read_csv(
    "results/step_05/test_target_identities.csv"
)

identity_columns = [
    "Reference",
    "Origin_Date",
    "Target_Date",
    "Horizon",
    "Actual"
]

actual_identities = (
    predictions_df[identity_columns]
    .sort_values(["Reference", "Horizon"])
    .reset_index(drop=True)
)

expected_identities = (
    expected_targets[identity_columns]
    .sort_values(["Reference", "Horizon"])
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    actual_identities,
    expected_identities,
    check_dtype=False
)

assert len(predictions_df) == (
    n_parts * FORECAST_HORIZON
)


# 16. Calculate and save the overall test metrics
test_metrics = calculate_metrics(
    predictions_df["Actual"],
    predictions_df["Prediction"]
)

metrics_df = pd.DataFrame([
    {
        "Model": "TFT",
        **test_metrics
    }
])

metrics_df.to_csv(
    output_folder / "TFT_test_metrics.csv",
    index=False
)

predictions_df.to_csv(
    output_folder / "TFT_test_predictions.csv",
    index=False
)

print("\nTFT test metrics:")
print(
    metrics_df.round(4).to_string(index=False)
)


# 17. Calculate metrics for each forecast horizon
horizon_records = []

for horizon, group in predictions_df.groupby("Horizon"):

    horizon_records.append({
        "Model": "TFT",
        "Horizon": int(horizon),
        **calculate_metrics(
            group["Actual"],
            group["Prediction"]
        )
    })

pd.DataFrame(
    horizon_records
).to_csv(
    output_folder / "TFT_metrics_by_horizon.csv",
    index=False
)


# 18. Verify decoder independence from future unknown values
sample_inputs, _ = next(
    iter(test_loader)
)

altered_inputs = copy.deepcopy(
    sample_inputs
)

for feature in unknown_reals:

    feature_index = (
        best_tft.hparams.x_reals.index(
            feature
        )
    )

    altered_inputs["decoder_cont"][
        :,
        :,
        feature_index
    ] = 123456.0

with torch.no_grad():

    original_output = best_tft(
        sample_inputs
    )["prediction"]

    altered_output = best_tft(
        altered_inputs
    )["prediction"]

torch.testing.assert_close(
    original_output,
    altered_output,
    rtol=0,
    atol=0
)

print(
    "\nPassed: unavailable future covariates "
    "do not affect TFT predictions."
)


# 19. Plot the available learning curves
history_df = pd.DataFrame(
    history_callback.records
)

fig, ax = plt.subplots(
    figsize=(8, 4)
)

if "train_loss_epoch" in history_df.columns:

    ax.plot(
        history_df["Epoch"],
        history_df["train_loss_epoch"],
        label="Training"
    )

elif "train_loss" in history_df.columns:

    ax.plot(
        history_df["Epoch"],
        history_df["train_loss"],
        label="Logged training loss"
    )

ax.plot(
    history_df["Epoch"],
    history_df["val_loss"],
    label="Validation"
)

ax.set_xlabel("Epoch")
ax.set_ylabel("Quantile loss")
ax.set_title("TFT learning curves — data")
ax.legend()

fig.tight_layout()

fig.savefig(
    output_folder / "TFT_learning_curves.png",
    dpi=160
)

plt.show()
plt.close(fig)


# 20. Plot actual versus predicted quantities
fig, ax = plt.subplots(
    figsize=(6, 6)
)

ax.scatter(
    predictions_df["Actual"],
    predictions_df["Prediction"],
    alpha=0.5,
    s=18
)

maximum_value = max(
    predictions_df["Actual"].max(),
    predictions_df["Prediction"].max(),
    1
)

ax.plot(
    [0, maximum_value],
    [0, maximum_value],
    color="black",
    linestyle="--",
    label="Perfect prediction"
)

ax.set_xlabel("Actual demand")
ax.set_ylabel("Predicted demand")
ax.set_title("TFT predictions ")
ax.legend()

fig.tight_layout()

fig.savefig(
    output_folder / "TFT_actual_vs_predicted.png",
    dpi=160
)

plt.show()
plt.close(fig)


# 21. Save the training configuration
configuration = {
    "model": "TemporalFusionTransformer",
    "hidden_size": HIDDEN_SIZE,
    "attention_heads": ATTENTION_HEADS,
    "hidden_continuous_size": 8,
    "dropout": 0.1,
    "learning_rate": LEARNING_RATE,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "optimizer": "Adam",
    "loss": "QuantileLoss",
    "quantiles": [0.1, 0.5, 0.9],
    "point_forecast": "Median",
    "encoder_length": ENCODER_LENGTH,
    "forecast_horizon": FORECAST_HORIZON,
    "seed": SEED,
    "device": "cpu",
    "best_checkpoint": str(best_checkpoint_path),
    "best_validation_loss": float(
        checkpoint_callback.best_model_score.item()
    ),
    "known_reals": known_reals,
    "historical_unknown_reals": unknown_reals,
    "test_metrics": test_metrics,
    "future_covariate_check_passed": True
}

(
    output_folder / "TFT_configuration.json"
).write_text(
    json.dumps(configuration, indent=2),
    encoding="utf-8"
)

print(f"\nResults saved in: {output_folder}")